## Выполнил: Воропаев Иван Константинович
### Группа: М8О-30Б-22

Импортируем библиотеки

In [1219]:
import optuna
import logging
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, PowerTransformer, PolynomialFeatures
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, HuberRegressor, BayesianRidge
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline
from feature_engine.selection import DropCorrelatedFeatures
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.feature_selection import RFECV
from sklearn.impute import KNNImputer
from sklearn.cluster import DBSCAN
from scipy.stats import skew
from pyod.models.pca import PCA
import warnings
warnings.filterwarnings("ignore")
logging.getLogger('optuna').setLevel(logging.WARNING)

Заранее качаем всё из Kaggle. Получаем архивчик mai-ml-contest-1. Его и спользуем

In [1220]:
loan_data = pd.read_csv('mai-ml-contest-1/train.csv')

Смотрим, что всё хорошо. Сверяем шапку, получаем основные показания данных, а также некоторые полезные данные, например, mean, min, max, количество данных, пустые строки и т.д.

In [1221]:
loan_data.head()
info = loan_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11017 entries, 0 to 11016
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ApplicationDate             10487 non-null  object 
 1   Age                         10487 non-null  float64
 2   AnnualIncome                10487 non-null  float64
 3   CreditScore                 9986 non-null   float64
 4   LoanAmount                  9986 non-null   float64
 5   LoanDuration                10487 non-null  float64
 6   MaritalStatus               10487 non-null  object 
 7   NumberOfDependents          10487 non-null  float64
 8   HomeOwnershipStatus         10487 non-null  object 
 9   MonthlyDebtPayments         9986 non-null   float64
 10  CreditCardUtilizationRate   10487 non-null  float64
 11  NumberOfOpenCreditLines     10487 non-null  float64
 12  NumberOfCreditInquiries     10487 non-null  float64
 13  DebtToIncomeRatio           104

In [1222]:
loan_data.describe()

,Age,AnnualIncome,CreditScore,LoanAmount,LoanDuration,NumberOfDependents,MonthlyDebtPayments,CreditCardUtilizationRate,NumberOfOpenCreditLines,NumberOfCreditInquiries,...,UtilityBillsPaymentHistory,JobTenure,Experience,NetWorth,BaseInterestRate,InterestRate,MonthlyLoanPayment,TotalDebtToIncomeRatio,LoanApproved,RiskScore
count,10487.000000,10487.000000,9986.000000,9986.000000,10487.000000,10487.000000,9986.000000,10487.000000,10487.000000,10487.000000,...,10487.000000,10487.000000,10487.000000,9.986000e+03,9986.000000,10487.000000,10487.000000,10487.000000,10487.000000,1.048700e+04
mean,39.850386,131587.872127,678.082716,29874.218306,53.439878,1.568323,546.458642,0.284397,3.033565,0.979498,...,0.784428,4.949271,17.628302,1.542381e+05,0.200392,0.200112,1075.622426,0.517577,0.511776,-2.569878e+04
std,11.614132,115791.941909,175.192486,27705.509722,24.493562,1.418684,501.981888,0.159240,1.740186,0.990927,...,0.123039,2.201100,11.337248,4.622229e+05,0.094388,0.096458,1344.053181,0.894637,0.499885,1.431675e+06
min,18.000000,15000.000000,300.000000,1063.000000,12.000000,0.000000,13.000000,0.003674,0.000000,0.000000,...,0.259301,0.000000,0.000000,1.004000e+03,0.052494,0.046445,30.008506,0.006064,0.000000,-9.999999e+06
25%,32.000000,20959.500000,550.000000,12658.000000,36.000000,0.000000,233.250000,0.158929,2.000000,0.000000,...,0.708475,3.000000,9.000000,7.252500e+03,0.119908,0.119548,375.872620,0.066734,0.000000,3.256475e+01
50%,40.000000,89015.000000,722.500000,21828.500000,48.000000,1.000000,398.000000,0.262229,3.000000,1.000000,...,0.803692,5.000000,17.000000,2.742950e+04,0.182023,0.180710,684.878529,0.178193,1.000000,4.411876e+01
75%,48.000000,257025.000000,850.000000,37158.000000,60.000000,3.000000,685.000000,0.391683,4.000000,2.000000,...,0.879312,6.000000,26.000000,1.241758e+05,0.264709,0.264880,1279.930203,0.637457,1.000000,6.535690e+01
max,80.000000,748508.000000,850.000000,418997.000000,120.000000,6.000000,10879.000000,0.914635,12.000000,6.000000,...,0.996573,17.000000,57.000000,1.126117e+07,0.722497,0.833647,29634.807816,24.383046,1.000000,1.000000e+07


In [1223]:
loan_data.isnull().sum()

ApplicationDate                530
Age                            530
AnnualIncome                   530
CreditScore                   1031
LoanAmount                    1031
LoanDuration                   530
MaritalStatus                  530
NumberOfDependents             530
HomeOwnershipStatus            530
MonthlyDebtPayments           1031
CreditCardUtilizationRate      530
NumberOfOpenCreditLines        530
NumberOfCreditInquiries        530
DebtToIncomeRatio              530
BankruptcyHistory             1031
LoanPurpose                   1031
PreviousLoanDefaults           530
PaymentHistory                 530
LengthOfCreditHistory          530
SavingsAccountBalance          530
CheckingAccountBalance        1031
TotalAssets                   1031
TotalLiabilities               530
MonthlyIncome                  530
UtilityBillsPaymentHistory     530
JobTenure                      530
EmploymentStatus               530
EducationLevel                 530
Experience          

In [1224]:
loan_data.duplicated().sum()

1016

In [1225]:
print(loan_data.nunique(), loan_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11017 entries, 0 to 11016
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ApplicationDate             10487 non-null  object 
 1   Age                         10487 non-null  float64
 2   AnnualIncome                10487 non-null  float64
 3   CreditScore                 9986 non-null   float64
 4   LoanAmount                  9986 non-null   float64
 5   LoanDuration                10487 non-null  float64
 6   MaritalStatus               10487 non-null  object 
 7   NumberOfDependents          10487 non-null  float64
 8   HomeOwnershipStatus         10487 non-null  object 
 9   MonthlyDebtPayments         9986 non-null   float64
 10  CreditCardUtilizationRate   10487 non-null  float64
 11  NumberOfOpenCreditLines     10487 non-null  float64
 12  NumberOfCreditInquiries     10487 non-null  float64
 13  DebtToIncomeRatio           104

Количество пустых значений и дубликатов незначительно. Значит, удаляяем все строки и дубликаты.

Cтроим красивые графики, так как, ну, умеем. Почему бы не построить (на самом деле они пригодятся в конце).

In [1226]:
loan_data.hist(color='Green', bins=20, figsize=(25,30))
plt.suptitle('Распределения данных по каждому столбцу', fontsize=20)
plt.show()

## Предпроцессинг данных

### Работа с пропусками

Для начала удаляем все пропуски. Я пытался импутировать, но вышло совсем не очень. Просто стираем и не паримся.

In [1227]:
loan_data = loan_data.dropna()
loan_data = loan_data.loc[(loan_data['RiskScore'] >= 0) & (loan_data['RiskScore'] <= 100)]
loan_data.drop_duplicates(inplace=True)
print(loan_data['RiskScore'].min(), loan_data['RiskScore'].max())

14.841417296887238 97.59724939432462


### Категории

Категории будем обрабатывать с помощью LabelEncoder. Просто выписываем колонки и кодируем. Парсим время.

In [1228]:
categorical_columns = ['ApplicationDate', 'EmploymentStatus', 'EducationLevel', 'LoanDuration',
'MaritalStatus', 'NumberOfDependents', 'HomeOwnershipStatus', 'NumberOfOpenCreditLines',
'NumberOfCreditInquiries', 'BankruptcyHistory', 'LoanPurpose','PreviousLoanDefaults',
'PaymentHistory', 'LengthOfCreditHistory', 'UtilityBillsPaymentHistory', 'JobTenure',
'TotalDebtToIncomeRatio', 'LoanApproved']

non_encoded_columns = loan_data[categorical_columns].apply(lambda col: col.dtype == 'object')

non_encoded_columns = non_encoded_columns[non_encoded_columns == True].index
print("Незакодированные категориальные колонки:", list(non_encoded_columns))

loan_data['ApplicationDate'] = pd.to_datetime(loan_data['ApplicationDate'])

loan_data['ApplicationYear'] = loan_data['ApplicationDate'].dt.year
loan_data['ApplicationMonth'] = loan_data['ApplicationDate'].dt.month
loan_data['ApplicationDay'] = loan_data['ApplicationDate'].dt.day

loan_data = loan_data.drop(columns=['ApplicationDate'])

categorical_columns.remove('ApplicationDate')

categorical_columns_to_encode = ['EmploymentStatus', 'EducationLevel', 'MaritalStatus', 
                                 'HomeOwnershipStatus', 'LoanPurpose']

label_encoder = LabelEncoder()

for col in categorical_columns_to_encode:
    loan_data[col] = label_encoder.fit_transform(loan_data[col])

Незакодированные категориальные колонки: ['ApplicationDate', 'EmploymentStatus', 'EducationLevel', 'MaritalStatus', 'HomeOwnershipStatus', 'LoanPurpose']


### Численные данные

Переходим к нормализации данных. Некоторые данные могут иметь большой разброс - сейчас построим графики и убедимся в этом. Нас интересуют numeric колонки.

In [1230]:
numerical_columns = loan_data.select_dtypes(include=[np.number]).drop(columns=['RiskScore'] + categorical_columns).columns


# Всего 22 колонки - поэтому соберём по 2 в ряд.
n_cols = 2 
n_rows = len(numerical_columns) // n_cols + 1

plt.figure(figsize=(15, 5 * n_rows))

for i, col_name in enumerate(numerical_columns, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.boxplot(y=loan_data[col_name])
    plt.title(col_name)

plt.tight_layout()

plt.show()

Для начала разберёмся с выбросами по RiskScore. Его я слегка уберу с помощью IForest. Contamination в моём случае - магическое число, выведенное опытным путём.

In [1231]:
previous_len = len(loan_data)
outlier_detector = IsolationForest(contamination=0.015, random_state=42)
loan_data['outlier'] = outlier_detector.fit_predict(loan_data[['RiskScore']])
loan_data = loan_data[loan_data['outlier'] != -1]

loan_data = loan_data.drop(columns=['outlier'])

print(f"Всего данных было {previous_len}. Количество выбросов удалено: {previous_len - len(loan_data)}")
y_regression = loan_data['RiskScore']

Всего данных было 9332. Количество выбросов удалено: 140


Залогорифмируем данные. Однако с этого момента начинается самое интересное - Optuna. Предоставим самой машине понять, какие фичи стоит залогорифмировать.

In [1232]:
def preprocess_data(loan_data, skew_threshold):
    skewness = loan_data[numerical_columns].apply(skew)

    columns_to_log = skewness[skewness > skew_threshold].index.tolist()
    
    for col in columns_to_log:
        loan_data[col] = np.log1p(loan_data[col])
    
    return loan_data, columns_to_log


def objective_skew_threshold(trial):
    skew_threshold = trial.suggest_uniform('skew_threshold', 0.01, 0.99)

    loan_data_logged, _ = preprocess_data(loan_data.copy(), skew_threshold)

    model_type = trial.suggest_categorical('model_type', ['lasso', 'ridge', 'elasticnet'])
    
    if model_type == 'lasso':
        alpha = trial.suggest_loguniform('lasso_alpha', 1e-6, 50)
        model = Lasso(alpha=alpha)
    elif model_type == 'ridge':
        alpha = trial.suggest_loguniform('ridge_alpha', 1e-6, 50)
        model = Ridge(alpha=alpha)
    else:
        alpha = trial.suggest_loguniform('elasticnet_alpha', 1e-6, 50)
        l1_ratio = trial.suggest_uniform('elasticnet_l1_ratio', 0.0, 1.0)
        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)

    scores = cross_val_score(model, loan_data_logged, y_regression, scoring='neg_mean_squared_error', cv=10)
    mse = -scores.mean()

    return mse



study_skew = optuna.create_study(direction='minimize')
study_skew.optimize(objective_skew_threshold, n_trials=200)

best_skew_threshold = study_skew.best_params['skew_threshold']
print(f"Лучший порог асимметрии: {best_skew_threshold}")

loan_data, columns_to_log = preprocess_data(loan_data, study_skew.best_params['skew_threshold'])
print("Логарифмируем колонки:", columns_to_log)

Лучший порог асимметрии: 0.32644166883806924
Логарифмируем колонки: ['AnnualIncome', 'LoanAmount', 'MonthlyDebtPayments', 'CreditCardUtilizationRate', 'DebtToIncomeRatio', 'SavingsAccountBalance', 'CheckingAccountBalance', 'TotalAssets', 'TotalLiabilities', 'MonthlyIncome', 'NetWorth', 'BaseInterestRate', 'InterestRate', 'MonthlyLoanPayment']


Тут ниже представлен код обработки выбросов, однако он увеличивал MSE. Оставлю просто так. Использовался DBSCAN и тюнился с помощью Optuna. Применялся только к тем колонкам, которые не были до этого залогорифмированы.

In [1233]:
# def replace_outliers_with_iqr(loan_data, columns_to_smooth_outliers):
#     for col in columns_to_smooth_outliers:
#         Q1 = loan_data[col].quantile(0.25)
#         Q3 = loan_data[col].quantile(0.75)
#         IQR = Q3 - Q1
#         lower_bound = Q1 - 1.5 * IQR
#         upper_bound = Q3 + 1.5 * IQR
#         
#         loan_data[col] = np.where(loan_data[col] < lower_bound, lower_bound, loan_data[col])
#         loan_data[col] = np.where(loan_data[col] > upper_bound, upper_bound, loan_data[col])
#     
#     return loan_data
# 
# 
# def smooth_outliers_dbscan(loan_data, columns_to_smooth_outliers, eps, min_samples):
#     dbscan = DBSCAN(eps=eps, min_samples=min_samples)
#     dbscan_labels = dbscan.fit_predict(loan_data[columns_to_smooth_outliers])
#     
#     loan_data['dbscan_label'] = dbscan_labels
#     
#     loan_data.loc[loan_data['dbscan_label'] == -1, columns_to_smooth_outliers] = replace_outliers_with_iqr(
#         loan_data[columns_to_smooth_outliers], columns_to_smooth_outliers
#     )
#     
#     loan_data = loan_data.drop(columns=['dbscan_label'])
#     
#     return loan_data
# 
# 
# def objective_dbscan(trial):
#     eps = trial.suggest_uniform('eps', 0.1, 2.0)
#     min_samples = trial.suggest_int('min_samples', 3, 15)  
#     
#     loan_data_smoothed = smooth_outliers_dbscan(loan_data.copy(), columns_to_smooth_outliers, eps, min_samples)
#     
#     model_type = trial.suggest_categorical('model_type', ['lasso', 'ridge', 'elasticnet'])
#     
#     if model_type == 'lasso':
#         alpha = trial.suggest_loguniform('lasso_alpha', 1e-6, 50)
#         model = Lasso(alpha=alpha)
#     elif model_type == 'ridge':
#         alpha = trial.suggest_loguniform('ridge_alpha', 1e-6, 50)
#         model = Ridge(alpha=alpha)
#     else:
#         alpha = trial.suggest_loguniform('elasticnet_alpha', 1e-6, 50)
#         l1_ratio = trial.suggest_uniform('elasticnet_l1_ratio', 0.0, 1.0)
#         model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
#     
#     scores = cross_val_score(model, loan_data_smoothed, y_regression, scoring='neg_mean_squared_error', cv=10)
#     mse = -scores.mean()
# 
#     return mse
# 
# 
# columns_to_smooth_outliers = list(set(numerical_columns) - set(columns_to_log))
# 
# study_dbscan = optuna.create_study(direction='minimize')
# study_dbscan.optimize(objective_dbscan, n_trials=300)
# 
# best_params = study_dbscan.best_params
# print(f"Лучшие параметры DBSCAN: eps={best_params['eps']}, min_samples={best_params['min_samples']}")
# 
# loan_data = smooth_outliers_dbscan(loan_data.copy(), columns_to_smooth_outliers, best_params['eps'], best_params['min_samples'])

Отлично. Мы закончили с предобработкой данных.

Вспоминаем графики из начала. Конечно, строили мы их не просто так. Вот они после всех преобразований. Выглядят сильно лучше.

In [1234]:
loan_data.hist(color='Green', bins=20, figsize=(25,30))
plt.suptitle('Распределения данных по каждому столбцу', fontsize=20)
plt.show()

Строим HeatMap. 

P.S. При каждом запуске программы мой Mac отправляется в Вальгаллу.

In [1235]:
plt.figure(figsize=(30, 20))
plt.title('Корреляция численных данных')
loan_data.corr(numeric_only=True)
sns.heatmap(loan_data.corr(numeric_only=True), annot=True, fmt=".2f")

## Обучение

### Выбор фичей

Мы построили график и знаем корреляции. Нам нужно отбросить некоторые фичи и отобрать более значимые. Для анализа корреляционных признаков будем использовать Optuna (как и для всего).

In [1236]:
def objective_ridge_(trial, X, y):
    alpha = trial.suggest_loguniform('alpha', 1e-6, 50)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    
    model = Ridge(alpha=alpha, fit_intercept=fit_intercept)
    score = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=10)
    mse = -score.mean()
    
    return mse


def objective_lasso_(trial, X, y):
    alpha = trial.suggest_loguniform('alpha', 1e-6, 50)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    
    model = Lasso(alpha=alpha, fit_intercept=fit_intercept)
    score = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=10)
    mse = -score.mean()
    
    return mse


def objective_elasticnet_(trial, X, y):
    alpha = trial.suggest_loguniform('alpha', 1e-6, 50)
    l1_ratio = trial.suggest_uniform('l1_ratio', 0.0, 1.0)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, fit_intercept=fit_intercept)
    score = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=5)
    mse = -score.mean()
    
    return mse

Ниже Optuna выбирает фичи, которые нужно оставить. Из двух наборов - корреляционных и всех в совокупности.

In [1237]:
scorer = make_scorer(mean_squared_error, greater_is_better=False)

X_regression = loan_data.drop(columns=['RiskScore'])
drop_corr = DropCorrelatedFeatures(method='pearson', threshold=0.8)
X_corr_cleaned = drop_corr.fit_transform(X_regression)
y_regression = loan_data['RiskScore']

dropped_features = set(X_regression.columns) - set(X_corr_cleaned.columns)

best_mse = float('inf')
corr_features = []

estimator_model = Lasso(alpha=0.02)
estimator_model_mse = float('inf')

for feature in dropped_features:
    X_with_feature = X_corr_cleaned.copy()
    X_with_feature[feature] = X_regression[feature]

    study = optuna.create_study(direction='minimize')

    study.optimize(lambda trial: objective_ridge_(trial, X_with_feature, y_regression), n_trials=200)
    ridge_best_mse = study.best_value
    ridge_best_params = study.best_params

    study.optimize(lambda trial: objective_lasso_(trial, X_with_feature, y_regression), n_trials=200)
    lasso_best_mse = study.best_value
    lasso_best_params = study.best_params

    study.optimize(lambda trial: objective_elasticnet_(trial, X_with_feature, y_regression), n_trials=200)
    elasticnet_best_mse = study.best_value
    elasticnet_best_params = study.best_params
    
    avg_mse = min(ridge_best_mse, lasso_best_mse, elasticnet_best_mse)
    
    if avg_mse < best_mse:
        if min(ridge_best_mse, lasso_best_mse, elasticnet_best_mse) == ridge_best_mse:
            estimator_model = Ridge(alpha=ridge_best_params['alpha'])
        elif min(ridge_best_mse, lasso_best_mse, elasticnet_best_mse) == lasso_best_mse:
            estimator_model = Lasso(alpha=lasso_best_params['alpha'])
        else:
            estimator_model = ElasticNet(alpha=elasticnet_best_params['alpha'], l1_ratio=elasticnet_best_params['l1_ratio'])
        best_mse = avg_mse
        estimator_model_mse = best_mse
        print(f"Best MSE:{best_mse}")
        corr_features.append(feature)
        
print("Лучшие коррелированные признаки:", corr_features)

def objective_feature_selection(trial):
    selected_features = []

    for feature in X_regression.columns:
        if trial.suggest_categorical(f'use_feature_{feature}', [True, False]):
            selected_features.append(feature)
    
    if len(selected_features) == 0:
        return float('inf')
    
    X_selected = X_regression[selected_features]
    
    model_type = trial.suggest_categorical('model_type', ['lasso', 'ridge', 'elasticnet'])
    if model_type == 'lasso':
        alpha = trial.suggest_loguniform('lasso_alpha', 1e-6, 50)
        model = Lasso(alpha=alpha)
    elif model_type == 'ridge':
        alpha = trial.suggest_loguniform('ridge_alpha', 1e-6, 50)
        model = Ridge(alpha=alpha)
    else:
        alpha = trial.suggest_loguniform('elasticnet_alpha', 1e-6, 50)
        l1_ratio = trial.suggest_uniform('elasticnet_l1_ratio', 0.0, 1.0)
        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)

    scores = cross_val_score(model, X_selected, y_regression, scoring='neg_mean_squared_error', cv=10)
    mse = -scores.mean()

    return mse


study_features = optuna.create_study(direction='minimize')
study_features.optimize(objective_feature_selection, n_trials=300)

best_trial = study_features.best_trial
best_params = best_trial.params
optuna_selected_features = [feature for feature in X_regression.columns if best_params.get(f'use_feature_{feature}', False)]

print("Лучшие признаки Optuna:", optuna_selected_features)
print(f"Лучший MSE: {best_trial.value}")
print(f"Estimator model: {estimator_model}")

Best MSE:15.4945233361691
Лучшие коррелированные признаки: ['CreditScore']
Лучшие признаки Optuna: ['AnnualIncome', 'CreditScore', 'LoanAmount', 'LoanDuration', 'CreditCardUtilizationRate', 'NumberOfCreditInquiries', 'DebtToIncomeRatio', 'BankruptcyHistory', 'LoanPurpose', 'PreviousLoanDefaults', 'LengthOfCreditHistory', 'TotalAssets', 'UtilityBillsPaymentHistory', 'EmploymentStatus', 'NetWorth', 'BaseInterestRate', 'InterestRate', 'LoanApproved', 'ApplicationMonth']
Лучший MSE: 15.370967418335207
Estimator model: Ridge(alpha=0.08567681709928646)


Одной Optuna не ограничимся. Вызываем ещё один селектор SFS, который тоже отберёт нам фичи. Тюним его с помощью Optuna.

In [1238]:
def objective_sfs(trial):
    model_type = trial.suggest_categorical('model_type', ['lasso', 'ridge', 'elasticnet'])
    
    if model_type == 'lasso':
        alpha = trial.suggest_loguniform('lasso_alpha', 1e-6, 50)
        estimator_model = Lasso(alpha=alpha)
    elif model_type == 'ridge':
        alpha = trial.suggest_loguniform('ridge_alpha', 1e-6, 50)
        estimator_model = Ridge(alpha=alpha)
    else:
        alpha = trial.suggest_loguniform('elasticnet_alpha', 1e-6, 50)
        l1_ratio = trial.suggest_uniform('elasticnet_l1_ratio', 0.0, 1.0)
        estimator_model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
    
    sfs = SFS(estimator_model, 
              k_features='best', 
              forward=True, 
              floating=False, 
              scoring='neg_mean_squared_error', 
              cv=10)
    
    sfs.fit(X_regression, y_regression)
    sfs_selected_features = list(sfs.k_feature_names_)
    
    X_selected = X_regression[sfs_selected_features]
    scores = cross_val_score(estimator_model, X_selected, y_regression, scoring='neg_mean_squared_error', cv=10)
    mse = -scores.mean()
    
    return mse


study_sfs = optuna.create_study(direction='minimize')
study_sfs.optimize(objective_sfs, n_trials=50)

best_params = study_sfs.best_params
best_model_type = best_params['model_type']

if best_model_type == 'lasso':
    best_alpha = best_params['lasso_alpha']
    final_estimator_model = Lasso(alpha=best_alpha)
elif best_model_type == 'ridge':
    best_alpha = best_params['ridge_alpha']
    final_estimator_model = Ridge(alpha=best_alpha)
else:
    best_alpha = best_params['elasticnet_alpha']
    best_l1_ratio = best_params['elasticnet_l1_ratio']
    final_estimator_model = ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio)

sfs_best = SFS(
    final_estimator_model, 
    k_features='best', 
    forward=True, 
    floating=False, 
    scoring='neg_mean_squared_error', 
    cv=10
)
sfs_best.fit(X_regression, y_regression)
sfs_selected_features = list(sfs_best.k_feature_names_)
print("Лучшие признаки SFS:", sfs_selected_features)

Лучшие признаки SFS: ['AnnualIncome', 'CreditScore', 'LoanAmount', 'HomeOwnershipStatus', 'CreditCardUtilizationRate', 'DebtToIncomeRatio', 'BankruptcyHistory', 'PreviousLoanDefaults', 'LengthOfCreditHistory', 'TotalAssets', 'MonthlyIncome', 'EmploymentStatus', 'NetWorth', 'BaseInterestRate', 'MonthlyLoanPayment', 'TotalDebtToIncomeRatio', 'LoanApproved']


Но и это не всё - третий селектор - RFECV. 

In [1239]:
def objective_rfecv(trial):
    model_type = trial.suggest_categorical('model_type', ['lasso', 'ridge', 'elasticnet'])
    
    if model_type == 'lasso':
        alpha = trial.suggest_loguniform('lasso_alpha', 1e-6, 50)
        estimator_model = Lasso(alpha=alpha)
    elif model_type == 'ridge':
        alpha = trial.suggest_loguniform('ridge_alpha', 1e-6, 50)
        estimator_model = Ridge(alpha=alpha)
    else:
        alpha = trial.suggest_loguniform('elasticnet_alpha', 1e-6, 50)
        l1_ratio = trial.suggest_uniform('elasticnet_l1_ratio', 0.0, 1.0)
        estimator_model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
    
    rfecv = RFECV(estimator=estimator_model, step=1, cv=10, scoring='neg_mean_squared_error')
    rfecv.fit(X_regression, y_regression)
    
    X_selected = X_regression.loc[:, rfecv.support_]
    scores = cross_val_score(estimator_model, X_selected, y_regression, scoring='neg_mean_squared_error', cv=7)
    mse = -scores.mean()
    
    return mse


study_rfecv = optuna.create_study(direction='minimize')
study_rfecv.optimize(objective_rfecv, n_trials=70)

best_params = study_rfecv.best_params
best_model_type = best_params['model_type']

if best_model_type == 'lasso':
    best_alpha = best_params['lasso_alpha']
    final_estimator_model = Lasso(alpha=best_alpha)
elif best_model_type == 'ridge':
    best_alpha = best_params['ridge_alpha']
    final_estimator_model = Ridge(alpha=best_alpha)
else:
    best_alpha = best_params['elasticnet_alpha']
    best_l1_ratio = best_params['elasticnet_l1_ratio']
    final_estimator_model = ElasticNet(alpha=best_alpha, l1_ratio=best_l1_ratio)

rfecv_best = RFECV(estimator=final_estimator_model, step=1, cv=10, scoring='neg_mean_squared_error')
rfecv_best.fit(X_regression, y_regression)
rfecv_selected_features = X_regression.columns[rfecv_best.support_]
print("Лучшие признаки RFECV:", rfecv_selected_features)

Лучшие признаки RFECV: Index(['Age', 'AnnualIncome', 'CreditScore', 'LoanAmount',
       'HomeOwnershipStatus', 'MonthlyDebtPayments',
       'CreditCardUtilizationRate', 'DebtToIncomeRatio', 'BankruptcyHistory',
       'PreviousLoanDefaults', 'LengthOfCreditHistory', 'TotalAssets',
       'MonthlyIncome', 'UtilityBillsPaymentHistory', 'EmploymentStatus',
       'Experience', 'NetWorth', 'BaseInterestRate', 'InterestRate',
       'MonthlyLoanPayment', 'TotalDebtToIncomeRatio', 'LoanApproved'],
      dtype='object')


Итого, я отобрал фичи согласно трём селекторам. Все отобранные фичи отправляются для тренировки моделей. На этом шаге вызываем Scaler и отправляем на обучение.

In [1240]:
combined_selected_features = (set(optuna_selected_features) | set(corr_features)) | (set(sfs_selected_features) | set(rfecv_selected_features))
print(f"Признаки, которые остались после объединения: {combined_selected_features}")

dropped_features = set(X_regression.columns) - combined_selected_features
print("Удаляем признаки:", dropped_features)
X_regression = X_regression.drop(columns=dropped_features)

numerical_features = [col for col in X_regression.columns if col not in categorical_columns]

scaler = PowerTransformer()

X_regression[numerical_features] = scaler.fit_transform(X_regression[numerical_features])

kf = KFold(n_splits=4, shuffle=True, random_state=42)

Признаки, которые остались после объединения: {'TotalDebtToIncomeRatio', 'LoanDuration', 'TotalAssets', 'CreditCardUtilizationRate', 'LoanAmount', 'ApplicationMonth', 'AnnualIncome', 'UtilityBillsPaymentHistory', 'BankruptcyHistory', 'EmploymentStatus', 'BaseInterestRate', 'MonthlyLoanPayment', 'LoanPurpose', 'Age', 'LengthOfCreditHistory', 'HomeOwnershipStatus', 'NetWorth', 'InterestRate', 'PreviousLoanDefaults', 'MonthlyDebtPayments', 'Experience', 'NumberOfCreditInquiries', 'DebtToIncomeRatio', 'MonthlyIncome', 'LoanApproved', 'CreditScore'}
Удаляем признаки: {'ApplicationYear', 'NumberOfDependents', 'PaymentHistory', 'MaritalStatus', 'CheckingAccountBalance', 'NumberOfOpenCreditLines', 'EducationLevel', 'TotalLiabilities', 'JobTenure', 'SavingsAccountBalance', 'ApplicationDay'}


### Выбор модели

Обучим несколько видов моделей и выберем лучшую. Реализуем перебор гиперпараметров Optuna по нужной нам MSE. 

In [1241]:
poly_ridge_pipeline = Pipeline([
    ('polynomialfeatures', PolynomialFeatures()),
    ('ridge', Ridge())
])

poly_lasso_pipeline = Pipeline([
    ('polynomialfeatures', PolynomialFeatures()),
    ('lasso', Lasso())
])

poly_elasticnet_pipeline = Pipeline([
    ('polynomialfeatures', PolynomialFeatures()),
    ('elasticnet', ElasticNet())
])

poly_huber_pipeline = Pipeline([
    ('polynomialfeatures', PolynomialFeatures()),
    ('huber', HuberRegressor())
])

models = []

### Линейная регрессия

Просто учим, используя KFold. Лучшую модель сохраняем.

In [1242]:
best_linear = []
for train_index, test_index in kf.split(X_regression):
    Xr_train, Xr_test = X_regression.iloc[train_index], X_regression.iloc[test_index]
    yr_train, yr_test = y_regression.iloc[train_index], y_regression.iloc[test_index]
    linear=LinearRegression()
    linear.fit(Xr_train,yr_train)
    linear_mse = mean_squared_error(yr_test,linear.predict(Xr_test))
    best_linear.append([linear, linear_mse])

models.append(best_linear)

### Ridge

Начиная с этой модели мы будем организовывать перебор параметров Optuna. И сохранять лучшую модель аналогично. Код буквально под копирку.

In [1243]:
best_ridge = []
def objective_ridge(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-6, 50)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    
    model = Ridge(alpha=alpha, fit_intercept=fit_intercept)
    
    mse = cross_val_score(model, X_regression, y_regression, scoring='neg_mean_squared_error', cv=10).mean()
    
    return -mse

study_ridge = optuna.create_study(direction='minimize')
study_ridge.optimize(objective_ridge, n_trials=200)

best_ridge.append([Ridge(study_ridge.best_params), study_ridge.best_value])
    
models.append(best_ridge)

print(f"Лучшие гиперпараметры для Ridge: {study_ridge.best_params}")
print(f"Лучший MSE для Ridge: {study_ridge.best_value}")

Лучшие гиперпараметры для Ridge: {'alpha': 1.7528072890536053, 'fit_intercept': True}
Лучший MSE для Ridge: 14.50031765431927


### Lasso

In [1244]:
best_lasso = []
def objective_lasso(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-6, 50)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    
    model = Lasso(alpha=alpha, fit_intercept=fit_intercept)
    
    mse = cross_val_score(model, X_regression, y_regression, scoring='neg_mean_squared_error', cv=10).mean()
    
    return -mse


study_lasso = optuna.create_study(direction='minimize')
study_lasso.optimize(objective_lasso, n_trials=200)

best_lasso.append([Lasso(study_lasso.best_params), study_lasso.best_value])
    
models.append(best_lasso)

print(f"Лучшие гиперпараметры для Lasso: {study_lasso.best_params}")
print(f"Лучший MSE для Lasso: {study_lasso.best_value}")

Лучшие гиперпараметры для Lasso: {'alpha': 0.0005513941637538029, 'fit_intercept': True}
Лучший MSE для Lasso: 14.500583005721065


### ElasticNet

In [1245]:
best_elastic_net = []
def objective_elasticnet(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-6, 50)
    l1_ratio = trial.suggest_uniform('l1_ratio', 0.0, 1.0)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, fit_intercept=fit_intercept)
    
    mse = cross_val_score(model, X_regression, y_regression, scoring='neg_mean_squared_error', cv=10).mean()
    
    return -mse


study_elasticnet = optuna.create_study(direction='minimize')
study_elasticnet.optimize(objective_elasticnet, n_trials=300)

best_elastic_net.append([ElasticNet(study_elasticnet.best_params), study_elasticnet.best_value])
    
models.append(best_elastic_net)

print(f"Лучшие гиперпараметры для ElasticNet: {study_elasticnet.best_params}")
print(f"Лучший MSE для ElasticNet: {study_elasticnet.best_value}")

Лучшие гиперпараметры для ElasticNet: {'alpha': 0.00021602682337686354, 'l1_ratio': 0.03894454936412871, 'fit_intercept': True}
Лучший MSE для ElasticNet: 14.50027282495426


### Huber

P.S. Добавил, чтобы было - справляется хуже всех, а учится дольше того же Ridge. 

In [1246]:
best_huber = []
def objective_huber(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-6, 50)
    epsilon = trial.suggest_uniform('epsilon', 1.0, 9.9)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])

    model = HuberRegressor(alpha=alpha, epsilon=epsilon, fit_intercept=fit_intercept)

    mse = cross_val_score(model, X_regression, y_regression, scoring='neg_mean_squared_error', cv=10).mean()

    return -mse


study_huber = optuna.create_study(direction='minimize')
study_huber.optimize(objective_huber, n_trials=200)

best_huber.append([HuberRegressor(alpha=study_huber.best_params['alpha'], epsilon=study_huber.best_params['epsilon'], fit_intercept=study_huber.best_params['fit_intercept']), study_huber.best_value])

models.append(best_huber)

print(f"Лучшие гиперпараметры для Huber: {study_huber.best_params}")
print(f"Лучший MSE для Huber: {study_huber.best_value}")

Лучшие гиперпараметры для Huber: {'alpha': 49.527364913080646, 'epsilon': 2.636173783500631, 'fit_intercept': True}
Лучший MSE для Huber: 19.605033401301732


### Bayesian Ridge

In [1247]:
best_bayesian_ridge = []
def objective_bayesian_ridge(trial):
    alpha_1 = trial.suggest_loguniform('alpha_1', 1e-6, 50)
    alpha_2 = trial.suggest_loguniform('alpha_2', 1e-6, 50)
    lambda_1 = trial.suggest_loguniform('lambda_1', 1e-6, 50)
    lambda_2 = trial.suggest_loguniform('lambda_2', 1e-6, 50)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    
    model = BayesianRidge(alpha_1=alpha_1, alpha_2=alpha_2, lambda_1=lambda_1, lambda_2=lambda_2, fit_intercept=fit_intercept)
    
    mse = cross_val_score(model, X_regression, y_regression, scoring='neg_mean_squared_error', cv=10).mean()
    
    return -mse

study_bayesian_ridge = optuna.create_study(direction='minimize')
study_bayesian_ridge.optimize(objective_bayesian_ridge, n_trials=300)

best_bayesian_ridge.append([BayesianRidge(**study_bayesian_ridge.best_params), study_bayesian_ridge.best_value])
    
models.append(best_bayesian_ridge)

print(f"Лучшие гиперпараметры для Bayesian Ridge: {study_bayesian_ridge.best_params}")
print(f"Лучший MSE для Bayesian Ridge: {study_bayesian_ridge.best_value}")

Лучшие гиперпараметры для Bayesian Ridge: {'alpha_1': 13.998019376259993, 'alpha_2': 49.363213554262636, 'lambda_1': 15.749831329132439, 'lambda_2': 2.7202896905424508e-05, 'fit_intercept': True}
Лучший MSE для Bayesian Ridge: 14.500371491938699


### Полиномиальные фичи

В этих вариациях используются те же самые линейные модели, однако ещё и с полиномиальными фичами. Следовательно, обучение будет сильно дольше, но и результат меня очень радует.

### Ridge

In [1248]:
best_poly_ridge = []
def objective_poly_ridge(trial):
    degree = trial.suggest_int('polynomialfeatures__degree', 2, 2)
    alpha = trial.suggest_loguniform('ridge__alpha', 1e-6, 50)
    fit_intercept = trial.suggest_categorical('ridge__fit_intercept', [True, False])
    
    pipeline = Pipeline([
        ('polynomialfeatures', PolynomialFeatures(degree=degree)),
        ('ridge', Ridge(alpha=alpha, fit_intercept=fit_intercept))
    ])
    
    mse = cross_val_score(pipeline, X_regression, y_regression, scoring='neg_mean_squared_error', cv=10).mean()
    
    return -mse


study_poly_ridge = optuna.create_study(direction='minimize')
study_poly_ridge.optimize(objective_poly_ridge, n_trials=300)

final_model_ridge = Pipeline([
    ('polynomialfeatures', PolynomialFeatures(degree=2)),
    ('ridge', Ridge(
        alpha=study_poly_ridge.best_params['ridge__alpha'],
        fit_intercept=study_poly_ridge.best_params['ridge__fit_intercept']
    ))
])

final_model_ridge.fit(X_regression, y_regression)

best_poly_ridge.append([final_model_ridge, study_poly_ridge.best_value])

models.append(best_poly_ridge)

print(f"Лучшие гиперпараметры для полиномиального Ridge: {study_poly_ridge.best_params}")
print(f"Лучший MSE для полиномиального Ridge: {study_poly_ridge.best_value}")

Лучшие гиперпараметры для полиномиального Ridge: {'polynomialfeatures__degree': 2, 'ridge__alpha': 0.011770843599210743, 'ridge__fit_intercept': True}
Лучший MSE для полиномиального Ridge: 9.952062140471316


### Lasso

In [1249]:
best_poly_lasso = []
def objective_poly_lasso(trial):
    degree = trial.suggest_int('polynomialfeatures__degree', 2, 2)
    alpha = trial.suggest_loguniform('lasso__alpha', 1e-6, 50)
    fit_intercept = trial.suggest_categorical('lasso__fit_intercept', [True, False])

    pipeline = Pipeline([
        ('polynomialfeatures', PolynomialFeatures(degree=degree)),
        ('lasso', Lasso(alpha=alpha, fit_intercept=fit_intercept))
    ])

    mse = cross_val_score(pipeline, X_regression, y_regression, scoring='neg_mean_squared_error', cv=10).mean()

    return -mse


study_poly_lasso = optuna.create_study(direction='minimize')
study_poly_lasso.optimize(objective_poly_lasso, n_trials=200)

final_model_lasso = Pipeline([
    ('polynomialfeatures', PolynomialFeatures(degree=2)),
    ('lasso', Lasso(
        alpha=study_poly_lasso.best_params['lasso__alpha'],
        fit_intercept=study_poly_lasso.best_params['lasso__fit_intercept']
    ))
])

final_model_lasso.fit(X_regression, y_regression)

best_poly_lasso.append([final_model_lasso, study_poly_lasso.best_value])

models.append(best_poly_lasso)

print(f"Лучшие гиперпараметры для полиномиального Lasso: {study_poly_lasso.best_params}")
print(f"Лучший MSE для полиномиального Lasso: {study_poly_lasso.best_value}")

Лучшие гиперпараметры для полиномиального Lasso: {'polynomialfeatures__degree': 2, 'lasso__alpha': 0.0008866688132248499, 'lasso__fit_intercept': True}
Лучший MSE для полиномиального Lasso: 10.632517017857177


### ElasticNet

In [1250]:
best_poly_elasticnet = []
def objective_poly_elasticnet(trial):
    degree = trial.suggest_int('polynomialfeatures__degree', 2, 2)
    alpha = trial.suggest_loguniform('elasticnet__alpha', 1e-6, 50)
    l1_ratio = trial.suggest_uniform('elasticnet__l1_ratio', 0.0, 1.0)
    fit_intercept = trial.suggest_categorical('elasticnet__fit_intercept', [True, False])

    pipeline = Pipeline([
        ('polynomialfeatures', PolynomialFeatures(degree=degree)),
        ('elasticnet', ElasticNet(alpha=alpha, l1_ratio=l1_ratio, fit_intercept=fit_intercept))
    ])

    mse = cross_val_score(pipeline, X_regression, y_regression, scoring='neg_mean_squared_error', cv=10).mean()

    return -mse


study_poly_elasticnet = optuna.create_study(direction='minimize')
study_poly_elasticnet.optimize(objective_poly_elasticnet, n_trials=250)

final_model_elasticnet = Pipeline([
    ('polynomialfeatures', PolynomialFeatures(degree=2)),
    ('elasticnet', ElasticNet(
        alpha=study_poly_elasticnet.best_params['elasticnet__alpha'],
        l1_ratio=study_poly_elasticnet.best_params['elasticnet__l1_ratio'],
        fit_intercept=study_poly_elasticnet.best_params['elasticnet__fit_intercept']
    ))
])

final_model_elasticnet.fit(X_regression, y_regression)

best_poly_elasticnet.append([final_model_elasticnet, study_poly_elasticnet.best_value])

models.append(best_poly_elasticnet)

print(f"Лучшие гиперпараметры для полиномиального ElasticNet: {study_poly_elasticnet.best_params}")
print(f"Лучший MSE для полиномиального ElasticNet: {study_poly_elasticnet.best_value}")

Лучшие гиперпараметры для полиномиального ElasticNet: {'polynomialfeatures__degree': 2, 'elasticnet__alpha': 0.0008504988527955671, 'elasticnet__l1_ratio': 0.9997830292362396, 'elasticnet__fit_intercept': True}
Лучший MSE для полиномиального ElasticNet: 10.632954251284755


### Huber

P.S. Тут MSE совсем без комментариев.

In [1251]:
best_poly_huber = []
def objective_poly_huber(trial):
    degree = trial.suggest_int('polynomialfeatures__degree', 2, 2)
    alpha = trial.suggest_loguniform('huber__alpha', 1e-6, 50)
    epsilon = trial.suggest_uniform('epsilon', 1.0, 9.9)
    fit_intercept = trial.suggest_categorical('huber__fit_intercept', [True, False])

    pipeline = Pipeline([
        ('polynomialfeatures', PolynomialFeatures(degree=degree)),
        ('huber', HuberRegressor(alpha=alpha, epsilon=epsilon, fit_intercept=fit_intercept))
    ])

    mse = cross_val_score(pipeline, X_regression, y_regression, scoring='neg_mean_squared_error', cv=10).mean()

    return -mse


study_poly_huber = optuna.create_study(direction='minimize')
study_poly_huber.optimize(objective_poly_huber, n_trials=150)

final_model = Pipeline([
    ('polynomialfeatures', PolynomialFeatures(degree=2)),
    ('huber', HuberRegressor(
        alpha=study_poly_huber.best_params['huber__alpha'],
        epsilon=study_poly_huber.best_params['epsilon'],
        fit_intercept=study_poly_huber.best_params['huber__fit_intercept']
    ))
])

final_model.fit(X_regression, y_regression)

best_poly_huber.append([final_model, study_poly_huber.best_value])

models.append(best_poly_huber)

print(f"Лучшие гиперпараметры для полиномиального Huber: {study_poly_huber.best_params}")
print(f"Лучший MSE для полиномиального Huber: {study_poly_huber.best_value}")

Лучшие гиперпараметры для полиномиального Huber: {'polynomialfeatures__degree': 2, 'huber__alpha': 2.7066929196464287e-05, 'epsilon': 1.1476210345242048, 'huber__fit_intercept': True}
Лучший MSE для полиномиального Huber: 127.89582163961651


### BayesianRidge

In [1252]:
best_poly_bayesian_ridge = []
def objective_poly_bayesian_ridge(trial):
    degree = trial.suggest_int('polynomialfeatures__degree', 2, 2)
    alpha_1 = trial.suggest_loguniform('alpha_1', 1e-6, 50)
    alpha_2 = trial.suggest_loguniform('alpha_2', 1e-6, 50)
    lambda_1 = trial.suggest_loguniform('lambda_1', 1e-6, 50)
    lambda_2 = trial.suggest_loguniform('lambda_2', 1e-6, 50)
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    
    pipeline = Pipeline([
        ('polynomialfeatures', PolynomialFeatures(degree=degree)),
        ('bayesianridge', BayesianRidge(alpha_1=alpha_1, alpha_2=alpha_2, 
                                          lambda_1=lambda_1, lambda_2=lambda_2, 
                                          fit_intercept=fit_intercept))
    ])
    
    mse = cross_val_score(pipeline, X_regression, y_regression, scoring='neg_mean_squared_error', cv=10).mean()
    
    return -mse

study_poly_bayesian_ridge = optuna.create_study(direction='minimize')
study_poly_bayesian_ridge.optimize(objective_poly_bayesian_ridge, n_trials=300)

final_model_poly_bayesian_ridge = Pipeline([
    ('polynomialfeatures', PolynomialFeatures(degree=2)),
    ('bayesianridge', BayesianRidge(
        alpha_1=study_poly_bayesian_ridge.best_params['alpha_1'],
        alpha_2=study_poly_bayesian_ridge.best_params['alpha_2'],
        lambda_1=study_poly_bayesian_ridge.best_params['lambda_1'],
        lambda_2=study_poly_bayesian_ridge.best_params['lambda_2'],
        fit_intercept=study_poly_bayesian_ridge.best_params['fit_intercept']
    ))
])

final_model_poly_bayesian_ridge.fit(X_regression, y_regression)

best_poly_bayesian_ridge.append([final_model_poly_bayesian_ridge, study_poly_bayesian_ridge.best_value])

models.append(best_poly_bayesian_ridge)

print(f"Лучшие гиперпараметры для полиномиального Bayesian Ridge: {study_poly_bayesian_ridge.best_params}")
print(f"Лучший MSE для полиномиального Bayesian Ridge: {study_poly_bayesian_ridge.best_value}")

Лучшие гиперпараметры для полиномиального Bayesian Ridge: {'polynomialfeatures__degree': 2, 'alpha_1': 49.589439894225556, 'alpha_2': 7.07983544502891e-05, 'lambda_1': 0.00035572880425340506, 'lambda_2': 49.506699463672305, 'fit_intercept': False}
Лучший MSE для полиномиального Bayesian Ridge: 10.442623615862516


Выберем лучшую и с помощью неё заполним submissions в совсем уж ближайшем будущем. Для этого мы вспоминаем, что я специально сохранял лучшие вариации каждого типа моделей. А значит, мы просто переберём по MSE и вызовем лучшую.

In [1253]:
def find_best_model(models):
    best_modell = None
    best_mse = float('inf')
    
    for model_list in models:
        for model in model_list:
            if model[1] < best_mse:
                best_mse = model[1]
                best_modell = model[0]
    
    return [best_modell, best_mse]

best_model = find_best_model(models)

print(f"Лучшая модель: {best_model[0]} с MSE: {best_model[1]}")

Лучшая модель: Pipeline(steps=[('polynomialfeatures', PolynomialFeatures()),
                ('ridge', Ridge(alpha=0.011770843599210743))]) с MSE: 9.952062140471316


Лучшая - Poly Ridge. Выбираем её для предсказаний уже на 'test.csv'.

### Обработка данных для 'test.csv'

Всё это у нас замечательно работало для 'train.csv'. Однако сдавать нам нужно по 'test.csv'. А оно не обработано. Исправляем.

In [1254]:
data = pd.read_csv('mai-ml-contest-1/test.csv')

categorical_columns.append('ApplicationDate')

non_encoded_columns = data[categorical_columns].apply(lambda col: col.dtype == 'object')
non_encoded_columns = non_encoded_columns[non_encoded_columns == True].index

data['ApplicationDate'] = pd.to_datetime(data['ApplicationDate'])

data['ApplicationYear'] = data['ApplicationDate'].dt.year
data['ApplicationMonth'] = data['ApplicationDate'].dt.month
data['ApplicationDay'] = data['ApplicationDate'].dt.day

data = data.drop(columns=['ApplicationDate'])

categorical_columns.remove('ApplicationDate')

for col in categorical_columns_to_encode:
    data[col] = label_encoder.fit_transform(data[col])
    
data, columns_to_log = preprocess_data(data, study_skew.best_params['skew_threshold'])
print("Логарифмируем колонки:", columns_to_log)

# data = smooth_outliers_dbscan(data.copy(), columns_to_smooth_outliers, study_dbscan.best_params['eps'], study_dbscan.best_params['min_samples'])

Логарифмируем колонки: ['AnnualIncome', 'LoanAmount', 'MonthlyDebtPayments', 'CreditCardUtilizationRate', 'DebtToIncomeRatio', 'SavingsAccountBalance', 'CheckingAccountBalance', 'TotalAssets', 'TotalLiabilities', 'MonthlyIncome', 'NetWorth', 'BaseInterestRate', 'InterestRate', 'MonthlyLoanPayment']


## Предсказываем

Предсказываем по только что подготовленному 'test.csv' и создаём файл submission.

In [1255]:
data_cleaned = data.drop(columns=['ID'])
X_final = data_cleaned[X_regression.columns]

numerical_features = [col for col in X_final.columns if col not in categorical_columns]

X_final[numerical_features] = scaler.fit_transform(X_final[numerical_features])

Y_final = best_model[0].predict(X_final)
Y_final = np.clip(Y_final, 0, 100)

submission = pd.DataFrame({ 'ID': data['ID'], 'RiskScore': Y_final })
submission.to_csv('submission.csv', index=False)